# BMIN 5200 — Week 2 in-class exercise
## A propositional model checker, and the limits of truth tables

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week02.ipynb)

**Time:** ~25 minutes · **Pairs with:** Knowledge representation and propositional logic (truth tables, entailment, inference rules)

### What you'll do
- Represent logical sentences as nested Python tuples and write `evaluate(sentence, model)`
- Enumerate all 2^n models of a knowledge base and build a truth table from them
- Implement `entails(kb, query)` as a search for a counterexample, and run it on the surgical-abdomen KB from lecture
- Time the model checker as the KB grows and watch 2^n end the party

### Why it matters
Entailment is the precise version of a claim clinicians make constantly: *given what we know
about this patient, this conclusion follows.* A model checker is the most direct possible
implementation of that claim, and writing one takes about fifteen lines. It is also
completely unusable on a real problem list, and seeing exactly where it dies is what makes
the rest of the course — inference rules, ontologies, rule engines, Bayesian networks — look
like engineering rather than fashion.

## Setup

`sympy` is used only at the very end, to check your hand-rolled answer against a library
implementation. Everything before that is the Python standard library.

In [ ]:
%pip install -q sympy

import itertools
import time

import matplotlib.pyplot as plt
import pandas as pd

print("Setup complete.")

## Part 1 — A sentence is a tree

We will write logical sentences as nested tuples: the first element is the connective, the
rest are its arguments, and a bare string is a propositional symbol. So

`("->", ("and", "fever", "cough"), "flu")`

is *(fever ∧ cough) → flu*. A **model** is just a dict assigning True or False to every
symbol. `evaluate` walks the tree and returns the sentence's truth value in that model. Read
it — this is the whole semantics of propositional logic in twelve lines, and everything else
in this notebook is built on it.

In [ ]:
def evaluate(sentence, model):
    if isinstance(sentence, str):          # a bare symbol: look it up in the model
        return model[sentence]
    connective = sentence[0]
    if connective == "not":
        return not evaluate(sentence[1], model)
    if connective == "and":
        return all(evaluate(argument, model) for argument in sentence[1:])
    if connective == "or":
        return any(evaluate(argument, model) for argument in sentence[1:])
    if connective == "->":
        # Material implication: false only when the antecedent holds and the consequent fails.
        return (not evaluate(sentence[1], model)) or evaluate(sentence[2], model)
    if connective == "<->":
        return evaluate(sentence[1], model) == evaluate(sentence[2], model)
    raise ValueError(f"unknown connective: {connective}")


flu_rule = ("->", ("and", "fever", "cough"), "flu")

patient_a = {"fever": True, "cough": True, "flu": True}
patient_b = {"fever": True, "cough": True, "flu": False}

print("rule:", flu_rule)
print("holds for patient A (fever, cough, flu):     ", evaluate(flu_rule, patient_a))
print("holds for patient B (fever, cough, no flu):  ", evaluate(flu_rule, patient_b))

### Predict before you run

Patient C has **no fever, no cough, and no flu**. Does the sentence
*(fever ∧ cough) → flu* come out True or False for patient C? Commit to an answer out loud
before running the next cell — most people get this one wrong the first time, and the whole
notion of entailment depends on it.

In [ ]:
patient_c = {"fever": False, "cough": False, "flu": False}
print("holds for patient C (no fever, no cough, no flu):", evaluate(flu_rule, patient_c))

# The rule is a promise about what happens when fever and cough are both present.
# Patient C never triggers the promise, so the promise is not broken. A rule that
# never fires is not thereby falsified -- and it also tells you nothing.

## Part 2 — Every model, and the truth table

A KB with *n* symbols has 2^n models, and `itertools.product` generates them in one line.
`symbols_in` and `all_models` below are plumbing and are written for you. Your job is the
last line of `truth_table`: right now the result column is filled with `"?"`, so the cell
runs but tells you nothing. Replace it with the sentence's actual truth value in each model.

In [ ]:
def symbols_in(sentence):
    """Every propositional symbol appearing anywhere in a sentence."""
    if isinstance(sentence, str):
        return {sentence}
    found = set()
    for argument in sentence[1:]:
        found |= symbols_in(argument)
    return found


def all_symbols(sentences):
    found = set()
    for sentence in sentences:
        found |= symbols_in(sentence)
    return sorted(found)          # sorted so every laptop enumerates in the same order


def all_models(names):
    for values in itertools.product([False, True], repeat=len(names)):
        yield dict(zip(names, values))


def truth_table(sentence):
    names = all_symbols([sentence])
    rows = []
    for model in all_models(names):
        row = dict(model)
        # TODO: replace "?" with the truth value of `sentence` in this model.
        #       You already have the function that computes it.
        row["SENTENCE"] = "?"
        rows.append(row)
    return pd.DataFrame(rows)


appendicitis_rule = ("->", ("and", "abdominal_pain", "rebound_tenderness"), "suspect_appendicitis")
print(truth_table(appendicitis_rule).to_string(index=False))

Once the TODO is filled in, look at the rows where `abdominal_pain` is False. The rule is
True in every one of them, including the row where the patient has no pain, no rebound
tenderness, and we nonetheless suspect appendicitis. Material implication is weaker than the
clinical "if" we say out loud: it constrains only the rows where the antecedent fires. This
is the table from the *Computing Truth Tables* slide, and it is the reason a KB of
implications can be perfectly satisfied by a model that looks clinically absurd.

## Part 3 — Entailment is a failed search for a counterexample

KB entails a query when **there is no model that satisfies the KB and falsifies the query**.
That definition is directly executable: walk every model, and the first time you find one
where the KB holds but the query does not, you are done — the answer is no, and that model is
the counterexample. The skeleton below is complete except for the counterexample test. As
written it never finds one, so it claims everything is entailed. Fill in the TODO.

In [ ]:
def entails(kb, query):
    """Returns (True, None) if kb entails query, else (False, counterexample_model)."""
    names = all_symbols(list(kb) + [query])
    for model in all_models(names):
        if all(evaluate(sentence, model) for sentence in kb):
            # This model satisfies every sentence in the KB. It is a live possibility.
            # TODO: if the query is FALSE in this model, we have found a counterexample.
            #       Return (False, model) right here.
            pass
    return True, None


def show(label, kb, query):
    holds, counterexample = entails(kb, query)
    print(f"{label:34s} entailed: {holds}")
    if counterexample is not None:
        true_symbols = sorted(name for name, value in counterexample.items() if value)
        print(f"{'':34s} counterexample -- true in this model: {true_symbols}")


# The three inference rules from lecture, as a unit test. Two are valid, one is not.
show("modus ponens",
     [("->", "strep_positive", "prescribe_penicillin"), "strep_positive"],
     "prescribe_penicillin")

show("modus tollens",
     [("->", "strep_positive", "prescribe_penicillin"), ("not", "prescribe_penicillin")],
     ("not", "strep_positive"))

show("affirming the consequent",
     [("->", "strep_positive", "prescribe_penicillin"), "prescribe_penicillin"],
     "strep_positive")

If all three came back `True`, your counterexample test is still the placeholder — the third
one is the classic fallacy and must come back False, with a counterexample in which the
patient got penicillin without a positive strep test (for a sore throat that turned out to be
viral, say). Fix the TODO until modus ponens and modus tollens are valid and affirming the
consequent is not. That third line is the whole reason we test entailment mechanically
instead of by eye.

### The surgical abdomen KB

This is the knowledge base from the *Building a Truth Table: Surgical abdomen* slide, with
two extra rules so the pieces interact. It is synthetic and radically simplified — no real
appendicitis workup turns on three booleans. Ten symbols means 1,024 models, which your
checker will grind through in a few milliseconds.

**Predict before you run:** of the five queries below, how many are entailed? Specifically —
does this KB entail that we should order an ultrasound, and does it entail that we should
*not*?

In [ ]:
SURGICAL_ABDOMEN_KB = [
    # Rules
    ("->", ("and", "abdominal_pain", "rebound_tenderness", "elevated_wbc"), "suspect_appendicitis"),
    ("->", ("and", "abdominal_pain", "nausea", ("not", "rebound_tenderness")), "suspect_gastroenteritis"),
    ("->", "suspect_appendicitis", "surgical_abdomen"),
    ("->", ("and", "surgical_abdomen", ("not", "pregnant")), "order_ct"),
    ("->", ("and", "surgical_abdomen", "pregnant"), "order_ultrasound"),

    # Findings for the patient in front of us
    "abdominal_pain",
    "rebound_tenderness",
    "elevated_wbc",
    "nausea",
    ("not", "pregnant"),
]

print(f"{len(all_symbols(SURGICAL_ABDOMEN_KB))} symbols -> "
      f"{2 ** len(all_symbols(SURGICAL_ABDOMEN_KB))} models to check\n")

show("suspect_appendicitis", SURGICAL_ABDOMEN_KB, "suspect_appendicitis")
show("surgical_abdomen", SURGICAL_ABDOMEN_KB, "surgical_abdomen")
show("order_ct", SURGICAL_ABDOMEN_KB, "order_ct")
show("order_ultrasound", SURGICAL_ABDOMEN_KB, "order_ultrasound")
show("NOT order_ultrasound", SURGICAL_ABDOMEN_KB, ("not", "order_ultrasound"))

With the TODO filled in, `order_ultrasound` is not entailed **and neither is its negation**.
The KB never says what to do about ultrasound in a non-pregnant patient, so both possibilities
survive; asking the KB is not the same as asking the world. That gap is where the closed-world
assumption gets quietly smuggled into real clinical software — a system that reports "no
indication for ultrasound" when its KB is merely silent is making a claim its logic does not
support. We will meet this again in Week 3 with ontologies and in Week 6 with rule engines.

## Part 4 — Where this approach dies

The checker is correct. It is also enumerating every possible world, including worlds that
differ only in findings no rule mentions. Below, a three-sentence KB about a febrile patient
is padded with additional findings the rules never touch — each one is recorded only as
"present or absent", which is no information at all, and each one doubles the work.

Run it. Watch the seconds column.

In [ ]:
def padded_kb(n_symbols):
    """A tiny clinical chain, padded with findings that appear in no rule."""
    kb = [
        "fever_day1",
        ("->", "fever_day1", "fever_day2"),
        ("->", "fever_day2", "start_empiric_antibiotics"),
    ]
    for i in range(n_symbols - 3):
        # "the patient either has this finding or does not" -- true in every model,
        # informative in none, and it still doubles the number of models.
        kb.append(("or", f"unrelated_finding_{i}", ("not", f"unrelated_finding_{i}")))
    return kb


timings = []
for n_symbols in [10, 14, 18, 20]:
    kb = padded_kb(n_symbols)
    started = time.perf_counter()
    entails(kb, "start_empiric_antibiotics")
    elapsed = time.perf_counter() - started
    timings.append({"symbols": n_symbols, "models": 2 ** n_symbols, "seconds": round(elapsed, 3)})

print(pd.DataFrame(timings).to_string(index=False))

Four extra findings — one basic metabolic panel's worth of yes/no results — turned
milliseconds into seconds. Plot it on a log scale and the curve is a straight line, which is
what exponential growth looks like when you stop being able to argue with it.

In [ ]:
measured = pd.DataFrame(timings)

plt.figure(figsize=(6, 4))
plt.semilogy(measured["symbols"], measured["seconds"], marker="o")
plt.xlabel("propositional symbols in the KB")
plt.ylabel("seconds to answer one query (log scale)")
plt.title("Model checking: every new symbol doubles the work")
plt.grid(True, which="both", alpha=0.3)
plt.show()

# Extrapolate from the largest measurement we actually made.
models_per_second = measured["models"].iloc[-1] / measured["seconds"].iloc[-1]


def readable(seconds):
    for size, unit in [(1, "seconds"), (60, "minutes"), (3600, "hours"),
                       (86400, "days"), (86400 * 365, "years")]:
        if seconds < size * 1000:
            return f"{seconds / size:,.1f} {unit}"
    return f"{seconds / 86400 / 365:,.0f} years"


print(f"measured rate: {models_per_second:,.0f} models/second\n")
for n_symbols in [20, 30, 40, 60]:
    seconds = 2 ** n_symbols / models_per_second
    print(f"{n_symbols:3d} symbols -> {2 ** n_symbols:>22,} models -> {readable(seconds):>20s}")

A problem list, a medication list, and one panel of labs is well past 60 propositions, and
that is a single patient at a single moment. This is the *truth table method disadvantage*
slide, measured on your own laptop. Propositional entailment is co-NP-complete, so nobody
expects to fix this in general — but almost nothing clinical needs the general case. Modus
ponens and resolution get an answer by manipulating the sentences directly, never
constructing a model; forward chaining over Horn clauses (Week 6) runs in time linear in the
size of the KB. Everything after today is a representation chosen so that you never have to
enumerate what you are talking about.

### Checking the hand-rolled version against a library

Finally, the same query through `sympy`. If your `entails` disagrees with this, trust sympy
and go find your bug — but note that you now know exactly what sympy is doing, which you
would not if we had started here.

In [ ]:
import sympy
from sympy.logic.inference import entails as sympy_entails

(abdominal_pain, rebound_tenderness, elevated_wbc, nausea, pregnant,
 suspect_appendicitis, suspect_gastroenteritis, surgical_abdomen,
 order_ct, order_ultrasound) = sympy.symbols(
    "abdominal_pain rebound_tenderness elevated_wbc nausea pregnant "
    "suspect_appendicitis suspect_gastroenteritis surgical_abdomen "
    "order_ct order_ultrasound")

sympy_kb = [
    sympy.Implies(abdominal_pain & rebound_tenderness & elevated_wbc, suspect_appendicitis),
    sympy.Implies(abdominal_pain & nausea & ~rebound_tenderness, suspect_gastroenteritis),
    sympy.Implies(suspect_appendicitis, surgical_abdomen),
    sympy.Implies(surgical_abdomen & ~pregnant, order_ct),
    sympy.Implies(surgical_abdomen & pregnant, order_ultrasound),
    abdominal_pain, rebound_tenderness, elevated_wbc, nausea, ~pregnant,
]

comparison = []
for name, ours, theirs in [
    ("order_ct", "order_ct", order_ct),
    ("order_ultrasound", "order_ultrasound", order_ultrasound),
    ("suspect_appendicitis", "suspect_appendicitis", suspect_appendicitis),
]:
    mine, _ = entails(SURGICAL_ABDOMEN_KB, ours)
    comparison.append({"query": name, "our_checker": mine,
                       "sympy": sympy_entails(theirs, sympy_kb)})

print(pd.DataFrame(comparison).to_string(index=False))

## Talk about it

1. The KB entails neither `order_ultrasound` nor its negation. A clinical decision support
   tool has to display *something*. What should it display, and who decides — the knowledge
   engineer, the vendor, or the ordering physician?
2. We padded the KB with findings no rule mentions and the cost doubled each time. A real EHR
   contains thousands of facts about a patient that are irrelevant to the question being
   asked. What would a system need to know in order to decide what is irrelevant, and is that
   decision itself a logical one?
3. Every sentence in today's KB is certainly true or certainly false. Which of the five rules
   in the surgical abdomen KB would a surgeon actually sign off on as *always* true, and what
   representation would you need for the ones they would not?

## Solutions

Completed versions of the two TODOs, as markdown so that scrolling ahead does not overwrite
your work or slow the notebook down.

**Part 2 — `truth_table`:**

```python
        row["SENTENCE"] = evaluate(sentence, model)
```

**Part 3 — `entails`:**

```python
def entails(kb, query):
    names = all_symbols(list(kb) + [query])
    for model in all_models(names):
        if all(evaluate(sentence, model) for sentence in kb):
            if not evaluate(query, model):
                # One model is enough: the KB is satisfied and the query fails in it,
                # so the query does not follow from what we know.
                return False, model
    return True, None
```

Two details worth noticing in that function. First, it returns on the *first* counterexample,
so a query that fails is often answered quickly while a query that holds always costs the
full 2^n — proving something is expensive, refuting it can be cheap. Second, the symbols are
collected from the KB *and* the query together: a query mentioning a symbol the KB has never
heard of doubles the search space and is essentially never entailed, which is the logical
version of asking a question outside the system's competence.